In [ ]:
def main(datasources, start_date, end_date):
    if not isinstance(datasources, dict):
        raise TypeError('datasources must be a dictionary')
    if 'bar1m' not in datasources:
        raise KeyError('datasources must contain bar1m')
    if not isinstance(datasources['bar1m'], str) or not datasources['bar1m'].strip():
        raise ValueError('datasources bar1m must be a non-empty table identifier')
    import gc
    import os
    import json
    import math
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import dai
    FEATURES = ['ret_1m', 'legacy_vol_share', 'legacy_amt_share', 'legacy_tn_share', 'legacy_rel_trade_size', 'increment_vol_share', 'increment_amt_share', 'increment_tn_share', 'increment_rel_trade_size', 'ofi_l1', 'ofi_l5', 'depth_imb', 'depth_imb_l1', 'ord_imb', 'rel_spread', 'microprice_dev', 'slope_bid', 'slope_ask', 'close_in_range', 'intraday_cumret']

    def build_pool_sql(table, d0, d1):
        return f"\n        SELECT DISTINCT date::DATE AS pd, instrument\n        FROM bigalpha_2026_instruments\n        WHERE date::DATE BETWEEN DATE '{d0}' AND DATE '{d1}'\n        "

    def build_output_pool_sql(table, d0, d1):
        return 'SELECT pd AS date, instrument FROM (' + build_pool_sql(table, d0, d1) + ') p'

    def build_feature_sql(table, d0, d1, T):
        pool_sql = build_pool_sql(table, d0, d1)
        return f"\n        WITH pool AS (\n            {pool_sql}\n        ),\n        base AS (\n            SELECT\n                t.instrument, t.date, t.date::DATE AS d,\n                close, high, low, open,\n                volume::DOUBLE AS volume, amount::DOUBLE AS amount,\n                deal_number::DOUBLE AS deal_number,\n                ask_price1, bid_price1, ask_price5, bid_price5,\n                ask_volume1::DOUBLE AS av1, bid_volume1::DOUBLE AS bv1,\n                (bid_volume1+bid_volume2+bid_volume3+bid_volume4+bid_volume5)::DOUBLE AS bidv5,\n                (ask_volume1+ask_volume2+ask_volume3+ask_volume4+ask_volume5)::DOUBLE AS askv5,\n                (bid_num_orders1+bid_num_orders2+bid_num_orders3+bid_num_orders4+bid_num_orders5)::DOUBLE AS bidn5,\n                (ask_num_orders1+ask_num_orders2+ask_num_orders3+ask_num_orders4+ask_num_orders5)::DOUBLE AS askn5\n            FROM {table} t\n            INNER JOIN pool p ON p.instrument = t.instrument AND p.pd = t.date::DATE\n            WHERE t.date::DATE BETWEEN DATE '{d0}' AND DATE '{d1}'\n        ),\n        lagged AS (\n            SELECT *,\n                LAG(close)       OVER w AS close_p,\n                LAG(volume)      OVER w AS volume_p,\n                LAG(amount)      OVER w AS amount_p,\n                LAG(deal_number) OVER w AS deal_number_p,\n                LAG(bid_price1)  OVER w AS bp1_p,\n                LAG(ask_price1)  OVER w AS ap1_p,\n                LAG(bv1)         OVER w AS bv1_p,\n                LAG(av1)         OVER w AS av1_p,\n                LAG(bidv5)       OVER w AS bidv5_p,\n                LAG(askv5)       OVER w AS askv5_p,\n                FIRST_VALUE(open) OVER w_day AS day_open,\n                MAX(volume)       OVER (PARTITION BY instrument, d) AS peak_volume,\n                MAX(amount)       OVER (PARTITION BY instrument, d) AS peak_amount,\n                MAX(deal_number)  OVER (PARTITION BY instrument, d) AS peak_deal_number,\n                SUM(GREATEST(volume, 0))      OVER (PARTITION BY instrument, d) AS sum_volume,\n                SUM(GREATEST(amount, 0))      OVER (PARTITION BY instrument, d) AS sum_amount,\n                SUM(GREATEST(deal_number, 0)) OVER (PARTITION BY instrument, d) AS sum_deal_number,\n                ROW_NUMBER() OVER (PARTITION BY instrument, d ORDER BY date DESC) AS rn_desc\n            FROM base\n            WINDOW\n                w AS (PARTITION BY instrument, d ORDER BY date),\n                w_day AS (PARTITION BY instrument, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)\n        ),\n        flow AS (\n            SELECT *,\n                GREATEST(volume - COALESCE(volume_p, 0), 0) AS legacy_volume,\n                GREATEST(amount - COALESCE(amount_p, 0), 0) AS legacy_amount,\n                GREATEST(deal_number - COALESCE(deal_number_p, 0), 0) AS legacy_deal_number,\n                GREATEST(volume, 0) AS increment_volume,\n                GREATEST(amount, 0) AS increment_amount,\n                GREATEST(deal_number, 0) AS increment_deal_number,\n                (ask_price1 + bid_price1) / 2.0 AS mid\n            FROM lagged\n            WHERE rn_desc <= {T}\n        ),\n        feat AS (\n            SELECT\n                instrument, d, ({T} - rn_desc)::INT AS pos,\n                COALESCE(ln(NULLIF(close, 0) / NULLIF(close_p, 0)), 0) AS ret_1m,\n                COALESCE(legacy_volume / NULLIF(peak_volume, 0), 0) AS legacy_vol_share,\n                COALESCE(legacy_amount / NULLIF(peak_amount, 0), 0) AS legacy_amt_share,\n                COALESCE(legacy_deal_number / NULLIF(peak_deal_number, 0), 0) AS legacy_tn_share,\n                CASE WHEN legacy_volume > 0 AND legacy_deal_number > 0\n                           AND peak_volume > 0 AND peak_deal_number > 0\n                     THEN ln((legacy_volume / legacy_deal_number) / (peak_volume / peak_deal_number))\n                     ELSE 0 END AS legacy_rel_trade_size,\n                COALESCE(increment_volume / NULLIF(sum_volume, 0), 0) AS increment_vol_share,\n                COALESCE(increment_amount / NULLIF(sum_amount, 0), 0) AS increment_amt_share,\n                COALESCE(increment_deal_number / NULLIF(sum_deal_number, 0), 0) AS increment_tn_share,\n                CASE WHEN increment_volume > 0 AND increment_deal_number > 0\n                           AND sum_volume > 0 AND sum_deal_number > 0\n                     THEN ln((increment_volume / increment_deal_number) / (sum_volume / sum_deal_number))\n                     ELSE 0 END AS increment_rel_trade_size,\n                COALESCE(((CASE WHEN bid_price1 >= bp1_p THEN bv1 ELSE 0 END)\n                        - (CASE WHEN bid_price1 <= bp1_p THEN bv1_p ELSE 0 END)\n                        - (CASE WHEN ask_price1 <= ap1_p THEN av1 ELSE 0 END)\n                        + (CASE WHEN ask_price1 >= ap1_p THEN av1_p ELSE 0 END))\n                        / NULLIF(bv1 + av1 + bv1_p + av1_p, 0), 0) AS ofi_l1,\n                COALESCE(((bidv5 - bidv5_p) - (askv5 - askv5_p))\n                        / NULLIF(bidv5 + askv5 + bidv5_p + askv5_p, 0), 0) AS ofi_l5,\n                COALESCE((bidv5 - askv5) / NULLIF(bidv5 + askv5, 0), 0) AS depth_imb,\n                COALESCE((bv1 - av1) / NULLIF(bv1 + av1, 0), 0) AS depth_imb_l1,\n                COALESCE((bidn5 - askn5) / NULLIF(bidn5 + askn5, 0), 0) AS ord_imb,\n                COALESCE((ask_price1 - bid_price1) / NULLIF(mid, 0), 0) AS rel_spread,\n                COALESCE((ask_price1 * bv1 + bid_price1 * av1)\n                        / NULLIF(bv1 + av1, 0) / NULLIF(mid, 0) - 1, 0) AS microprice_dev,\n                COALESCE((bid_price1 - bid_price5) / NULLIF(mid, 0), 0) AS slope_bid,\n                COALESCE((ask_price5 - ask_price1) / NULLIF(mid, 0), 0) AS slope_ask,\n                CASE WHEN high > low THEN (close - low) / (high - low) ELSE 0.5 END AS close_in_range,\n                COALESCE(ln(NULLIF(close, 0) / NULLIF(day_open, 0)), 0) AS intraday_cumret\n            FROM flow\n        )\n        SELECT instrument, d, pos,\n               {', '.join((f'{column}::FLOAT AS {column}' for column in FEATURES))}\n        FROM feat\n        "

    def month_chunks(d0, d1):
        s, e = (pd.Timestamp(d0), pd.Timestamp(d1))
        out = []
        cur = s
        while cur <= e:
            me = min(cur + pd.offsets.MonthEnd(0), e)
            if me < cur:
                me = min(cur + pd.offsets.MonthEnd(1), e)
            out.append((cur.strftime('%Y-%m-%d'), me.strftime('%Y-%m-%d')))
            cur = me + pd.Timedelta(days=1)
        return out

    def frame_to_tensors(df, cfg):
        inst_codes, inst_uniq = pd.factorize(df['instrument'])
        day_codes, day_uniq = pd.factorize(pd.to_datetime(df['d']).values.astype('datetime64[D]'))
        combo = inst_codes.astype(np.int64) * len(day_uniq) + day_codes
        codes, uniq_combo = pd.factorize(combo)
        feats = df[FEATURES].to_numpy(dtype=np.float32, copy=True)
        np.nan_to_num(feats, copy=False, nan=0.0, posinf=10.0, neginf=-10.0)
        np.clip(feats, -10.0, 10.0, out=feats)
        x = np.zeros((len(uniq_combo), cfg.T, len(FEATURES)), dtype=np.float16)
        x[codes, df['pos'].to_numpy(dtype=np.int64)] = feats.astype(np.float16)
        cnt = np.bincount(codes, minlength=len(uniq_combo))
        keep = cnt >= cfg.MIN_BARS
        ci = (np.asarray(uniq_combo) // len(day_uniq)).astype(np.int64)
        cd = (np.asarray(uniq_combo) % len(day_uniq)).astype(np.int64)
        keys = pd.DataFrame({'instrument': np.asarray(inst_uniq)[ci], 'd': pd.to_datetime(day_uniq[cd])})
        return (keys[keep].reset_index(drop=True), x[keep])

    def fetch_features(dai_mod, table, d0, d1, cfg, day_sample):
        try:
            import psutil

            def _avail():
                return f'{psutil.virtual_memory().available / 2 ** 30:.1f}GB'
        except Exception:

            def _avail():
                return 'n/a'
        keys_all, xs = ([], [])
        chunks = month_chunks(d0, d1)
        for i, (cs, ce) in enumerate(chunks):
            sql = build_feature_sql(table, cs, ce, cfg.T)
            df = dai_mod.query(sql, filters={'date': [cs, ce + ' 23:59:59']}, compression=True).df()
            if day_sample > 1 and len(df):
                epoch_day = pd.to_datetime(df['d']).astype('int64') // 86400000000000
                df = df[epoch_day % day_sample == 0]
            if len(df) == 0:
                continue
            k, a = frame_to_tensors(df, cfg)
            del df
            gc.collect()
            k['chunk'] = len(xs)
            k['row'] = np.arange(len(k))
            keys_all.append(k)
            xs.append(a)
            print(f'[f002] 特征分块 {i + 1}/{len(chunks)} ({cs}..{ce}): {len(k)} stock-day | 剩余内存 {_avail()}')
        if not keys_all:
            raise ValueError(f'fetch_features: [{d0},{d1}] 无数据(表 {table})')
        return (pd.concat(keys_all, ignore_index=True), xs)

    class PatchTSTEncoder(nn.Module):

        def __init__(self, cfg):
            super().__init__()
            self.patch = cfg.PATCH
            self.n_patch = cfg.T // cfg.PATCH
            self.embed = nn.Linear(cfg.PATCH, cfg.D_MODEL)
            self.pos = nn.Parameter(torch.zeros(1, self.n_patch, cfg.D_MODEL))
            layer = nn.TransformerEncoderLayer(cfg.D_MODEL, cfg.N_HEADS, cfg.D_MODEL * 2, cfg.DROPOUT, batch_first=True, activation='gelu', norm_first=True)
            self.enc = nn.TransformerEncoder(layer, cfg.N_LAYERS)
            self.mix = nn.Linear(len(FEATURES) * cfg.D_MODEL, cfg.D_MODEL)
            self.norm = nn.LayerNorm(cfg.D_MODEL)

        def forward(self, x):
            b, _t, f = x.shape
            h = x.permute(0, 2, 1).reshape(b * f, self.n_patch, self.patch)
            h = self.enc(self.embed(h) + self.pos).mean(dim=1).reshape(b, -1)
            return self.norm(torch.nn.functional.gelu(self.mix(h)))

    class FactorModel(nn.Module):

        def __init__(self, name, cfg):
            super().__init__()
            if name != 'multitask_patchtst':
                raise ValueError(f'f002_multitask_oo_cc requires multitask_patchtst, got {name!r}')
            self.encoder = PatchTSTEncoder(cfg)
            self.dropout = nn.Dropout(cfg.DROPOUT)
            self.oo_head = nn.Linear(cfg.D_MODEL, 1)
            self.cc_head = nn.Linear(cfg.D_MODEL, 1)

        def forward_heads(self, x):
            hidden = self.dropout(self.encoder(x))
            return (self.oo_head(hidden).squeeze(-1), self.cc_head(hidden).squeeze(-1))

        def forward(self, x):
            _oo, cc = self.forward_heads(x)
            return cc

    def predict_factor(models, keys, xs, device, batch=1000, smooth=1):
        acc = np.zeros(len(keys), dtype=np.float64)
        for model in models:
            model.eval()
            out = np.empty(len(keys), dtype=np.float32)
            with torch.no_grad():
                for cid, g in keys.groupby('chunk'):
                    rows = g['row'].to_numpy()
                    pos = g.index.to_numpy()
                    xc = xs[int(cid)]
                    for i0 in range(0, len(rows), batch):
                        xb = torch.from_numpy(xc[rows[i0:i0 + batch]].astype(np.float32)).to(device)
                        out[pos[i0:i0 + batch]] = model(xb).cpu().numpy()
            z = pd.Series(out).groupby(keys['d'].to_numpy()).transform(lambda v: (v - v.mean()) / (v.std(ddof=0) + 1e-09))
            acc += z.to_numpy(np.float64)
        res = keys[['instrument', 'd']].copy()
        res['factor'] = acc / len(models)
        if smooth > 1:
            res = res.sort_values(['instrument', 'd']).reset_index(drop=True)
            res['factor'] = res.groupby('instrument')['factor'].transform(lambda s: s.rolling(smooth, min_periods=1).mean())
            res['factor'] = res.groupby('d')['factor'].transform(lambda v: (v - v.mean()) / (v.std(ddof=0) + 1e-09))
        return res

    class _InferCfg:

        def __init__(self, config):
            for key, value in config.items():
                setattr(self, key, value)

    def _find_weights(name):
        SAME_DIRECTORY_WEIGHT_ONLY = True
        if os.path.isabs(name):
            path = os.path.abspath(name)
        else:
            if name != 'f002_weights.json' or os.path.basename(name) != name:
                raise ValueError('relative weight name must be exactly f002_weights.json')
            path = os.path.join(os.getcwd(), 'f002_weights.json')
        if not os.path.isfile(path):
            raise FileNotFoundError('weight file is missing; package weights must be beside the submission notebook: ' + path)
        return path

    def run_inference(dai_mod, datasources, start_date, end_date, weights_name='f002_weights.json'):
        payload = json.load(open(_find_weights(weights_name), encoding='utf-8'))
        cfg = _InferCfg(payload['config'])
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = FactorModel(payload['arch'], cfg).to(device)
        model.load_state_dict({key: torch.tensor(np.asarray(value, dtype=np.float32)) for key, value in payload['state_dict'].items()})
        model.eval()
        start, end = (str(start_date)[:10], str(end_date)[:10])
        keys, arrays = fetch_features(dai_mod, datasources['bar1m'], start, end, cfg, 1)
        factor = predict_factor([model], keys, arrays, device).rename(columns={'d': 'date'})
        factor['date'] = pd.to_datetime(factor['date'])
        pool = dai_mod.query(build_output_pool_sql(datasources['bar1m'], start, end), filters={'date': [start_date, end_date]}, compression=True).df()
        pool['date'] = pd.to_datetime(pool['date'])
        factor = factor.merge(pool, on=['date', 'instrument'], how='inner')
        factor = factor.replace([np.inf, -np.inf], np.nan).dropna(subset=['factor'])
        return factor[['date', 'instrument', 'factor']]
    return run_inference(dai, datasources, start_date, end_date)
